## Surrogate vs. Thermochemical library
### **Time evolution analysis**
Import libraries

In [ ]:
import os
import sys
import numpy as np

In [ ]:
sys.path.append(os.path.abspath('./../functions'))

import utils
import error as err
import reading as read
import plotting as plt

%load_ext kernel_manager

Define inputs

In [ ]:
models = ['surrogate', 'thermolib']
scheme = 'strang'
nb_nodes = 701
nb_iters = 1100
iter_conv = 1000
iter_shots = [200, 400, 600, 'conv']
qoi = {
    'names': ['Y_O', 'Y_O2', 'Th', 'Tv'],
    'labels': {
        'Y_O2': (r"$Y_{\text{O}_2}$",),
        'Y_O': (r"$Y_{\text{O}}$",),
        'Th': (r"$T$", "[K]"),
        'Tv': (r"$T_{\text{v}}$", "[K]")
    }
}
path_to_case = "./../../"

In [ ]:
skip_flags = {
    'profiles': False,
    'error': False
}

Load data

In [ ]:
data = read.load_dataset(path_to_case, nb_nodes, nb_iters)

Global variables

In [ ]:
x = data['thermolib']['x']
xlim = [0.004, 0.007]

In [ ]:
time_shots = []
t = data['thermolib']['t']
for i in iter_shots:
    ti = t[iter_conv] if i == 'conv' else t[i]
    time_shots.append(ti.squeeze())
time_shots_lbl = [r'$t$ = ' + utils.sci_notation(ti,1) + ' s' for ti in time_shots]

In [ ]:
path_to_saving = os.path.abspath("./../output/time_evolve/")
os.makedirs(path_to_saving, exist_ok=True)

Shock capturing

In [ ]:
shock_loc = {}
for model in models:
    shock_loc[model] = {}
    for k in ('trans', 'conv'):
        shock_loc[model][k] = {}
        d = data[model][k]['u']
        for pos in ('start', 'end'):
            d_pos = np.flip(d, axis=1) if pos == 'end' else d
            idx = utils.capture_shock(d_pos)
            shock_loc[model][k][pos] = nb_nodes-1-idx if pos == 'end' else idx

Roll data

In [ ]:
ref_pos = shock_loc['thermolib']['trans']['start']
for model in models:
    for k, loc in shock_loc[model].items():
        shift = ref_pos[-1] if k == 'conv' else ref_pos
        shift = shift - loc['start']
        for q in qoi['names']:
            d = []
            for i, d_i in enumerate(data[model][k][q]):
                d.append(np.roll(d_i, shift[i]))
            data[model][k][q] = np.vstack(d)

### Profiles

In [ ]:
# 'x' transformation
xlbl = r"$x$ [mm]"
x = x * 1.e3
xlim = np.array(xlim) * 1.e3

Temperatures

In [ ]:
%%skip $skip_flags['profiles']

qois = ('Th', 'Tv')
labels = [xlbl, r"$T$ [K]"]

for i, it in enumerate(iter_shots):
    # Collect data
    if it == 'conv':
        regime = 'conv'
        it = -1
    else:
        regime = 'trans'
    y = np.vstack([data['thermolib'][regime][q][it] for q in qois]).T
    y_pred = np.vstack([data['surrogate'][regime][q][it] for q in qois]).T
    # Plot profiles
    plt.test(
        x,
        y,
        y_pred,
        var_names=[' '.join(qoi['labels'][q]) for q in qois],
        legend_loc='lower right',
        labels=labels,
        xlim=xlim,
        text=time_shots_lbl[i],
        figname=f'temp_{time_shots[i]}',
        path=path_to_saving
    )

Mass fractions

In [ ]:
%%skip $skip_flags['profiles']

qois = ('Y_O2', 'Y_O')
labels = [xlbl, r"$Y$"]

for i, it in enumerate(iter_shots):
    # Collect data
    if it == 'conv':
        regime = 'conv'
        it = -1
    else:
        regime = 'trans'
    y = np.vstack([data['thermolib'][regime][q][it] for q in qois]).T
    y_pred = np.vstack([data['surrogate'][regime][q][it] for q in qois]).T
    # Plot profiles
    ti = time_shots[i]
    plt.test(
        x,
        y,
        y_pred,
        var_names=[' '.join(qoi['labels'][q]) for q in qois],
        legend_loc='right',
        labels=labels,
        xlim=xlim,
        text=time_shots_lbl[i],
        figname=f'mass_{time_shots[i]}',
        path=path_to_saving
    )

Error

In [ ]:
%%skip $skip_flags['error']

e = {}
idx = np.array(list(shock_loc['thermolib']['trans'].values()))+3
for norm in ('mean', 'max'):
    e[norm] = {}
    for q in qoi['names']:
        e[norm][q] = []
        for it in iter_shots:
            # Collect data
            if it == 'conv':
                regime = 'conv'
                i = -1
            else:
                regime = 'trans'
                i = it
            e[norm][q].append(err.eval_err(
                y_pred=data['surrogate'][regime][q][i,idx[0][i]:idx[1][i]],
                y_true=data['thermolib'][regime][q][i,idx[0][i]:idx[1][i]],
                norm_fn=norm,
                err_fn='ape',
            ))

In [ ]:
%%skip $skip_flags['error']

ticks_lbl = [qoi['labels'][q][0] for q in e['mean'].keys()]
for norm, ei in e.items():
    ylabel = 'Mean error [\%]' if norm == 'mean' else 'Max error [\%]'
    plt.bars(
        list(ei.values()),
        ticks_lbl,
        width=0.13,
        var_names=time_shots_lbl,
        ylabel=ylabel,
        figname=f'error.{norm}.{len(ticks_lbl)}',
        path=path_to_saving
    )